In [ ]:
import torch
from models import TrainableModelBase
from dataclasses import dataclass
from torch import nn
from utils import *


class SimpleLinear(TrainableModelBase):
    @dataclass
    class ModelConfig(TrainableModelBase.ModelConfig):
        latent_dim: int = 32

    def __init__(
        self,
        user_matrix: torch.Tensor,
        item_matrix: torch.Tensor,
        rate_matrix: torch.Tensor,
        model_config: ModelConfig,
        loss_config: TrainableModelBase.LossConfig,
    ):
        super().__init__(model_config=model_config, loss_config=loss_config)
        self.rate_matrix = rate_matrix

        self.user_FS = nn.Linear(user_matrix.shape[1], self.model_config.latent_dim, bias=False)
        self.item_FS = nn.Linear(item_matrix.shape[1], self.model_config.latent_dim, bias=False)
    
    def forward(self, user_index: torch.Tensor, item_index: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        user_embeddings = self.user_FS(user_index)
        item_embeddings = self.item_FS(item_index)
        return user_embeddings, item_embeddings

    @torch.no_grad()
    def predict(self) -> torch.Tensor:
        user_embeddings = self.user_FS(self.rate_matrix)
        item_embeddings = self.item_FS(self.rate_matrix.T)
        return user_embeddings @ item_embeddings.T


In [ ]:
import matplotlib.pyplot as plt


def plot_metrics_by_set(test_result_list: list[dict], num_sets: int) -> None:
    if not test_result_list:
        return

    first_result = test_result_list[0]
    if not isinstance(first_result, dict):
        raise TypeError("Each test result should be a dict containing metric values.")

    # Keep numeric metrics only and plot the first four.
    metric_names = [
        key
        for key, value in first_result.items()
        if isinstance(value, (int, float))
    ][:4]

    if len(metric_names) < 4:
        raise ValueError("At least 4 numeric metrics are required for plotting.")

    x = list(range(1, num_sets + 1))

    plt.figure(figsize=(9, 5))
    for metric_name in metric_names:
        y = [result[metric_name] for result in test_result_list]
        plt.plot(x, y, marker="o", linewidth=2, label=metric_name)

    plt.xlabel("Set 编号")
    plt.ylabel("指标值")
    plt.title("不同 Set 的四项指标变化")
    plt.xticks(x)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


def d(
    user_matrix: torch.Tensor,
    item_matrix: torch.Tensor,
    rate_matrix: torch.Tensor,
    num_sets: int = 10,
    interaction_data: str = "games",
    semantic_data: str = "games",
):
    user_index_tuple = torch.chunk(torch.arange(user_matrix.shape[1]), num_sets)
    item_index_tuple = torch.chunk(torch.arange(item_matrix.shape[1]), num_sets)
    user_cumulative_index_list = []
    for i in range(len(user_index_tuple)):
        user_cumulative_index_list.append(torch.cat(user_index_tuple[: i + 1]))
    item_cumulative_index_list = []
    for i in range(len(item_index_tuple)):
        item_cumulative_index_list.append(torch.cat(item_index_tuple[: i + 1]))

    datahandler = DataHandler(
        interaction_data=interaction_data,
        semantic_data=semantic_data,
        batch_size=4096,
        num_neg_item=1,
    )
    metric = Metric(device="gpu")
    test_result_list = []

    for i in range(num_sets):
        model = SimpleLinear(
            user_matrix=user_matrix,
            item_matrix=item_matrix,
            rate_matrix=rate_matrix,
            model_config=SimpleLinear.ModelConfig(latent_dim=32),
            loss_config=TrainableModelBase.BPRLossConfig(
                similarity="dot",
                num_neg_item=1,
            ),
        )
        test_result = train_model(
            datahandler=datahandler,
            metric=metric,
            model=model,
        )
        test_result_list.append(test_result)

    plot_metrics_by_set(test_result_list=test_result_list, num_sets=num_sets)
    return test_result_list